### Goal: Prepare data for DBeaver import and subsequent SQL analysis.

### Main Steps:
* **Cleaning:** Handling non-standard formats (replacing delimiters, removing unnecessary characters).
* **Transformation:** Converting a single string with a list of items into a "one row — one item" format, which is essential for accurate SQL analysis.
* **Preparation:** Casting data types to the format required for MySQL (DBeaver) loading.

In [ ]:
import pandas as pd

# Load source data

df = pd.read_csv('receipts.csv')

# Replace commas with dots to handle decimal numbers correctly

df['Описание'] = df['Описание'].str.replace('0,19', '0.19', regex = False)
df['Описание'] = df['Описание'].str.replace('0,2', '0.2', regex = False)
df['Описание'] = df['Описание'].str.replace('0,3', '0.3', regex = False)
df['Описание'] = df['Описание'].str.replace('0,4', '0.4', regex = False)
df['Описание'] = df['Описание'].str.replace('0,5', '0.5', regex = False)

# Create a copy for further processing

df_temp = df.copy()

# Split the string by comma to create a list of items

df_temp['items'] = df['Описание'].str.split(',')

# Flatten the list into individual rows

df_final = df_temp.explode('items')

# Clean whitespace and quotes
df_final['items'] = df_final['items'].str.replace('"', "", regex = False).str.strip()

# Remove empty rows
df_final = df_final.dropna(subset=['items'])

# Split the item string into amount and product name
split_data = df_final['items'].str.split('x', n=1, expand=True)
df_final['item_amount'] = split_data[0]
df_final['item_prod'] = split_data[1]

# Trim whitespace and convert amount to numeric format
df_final['item_amount'] = df_final['item_amount'].str.strip()
df_final['item_prod'] = df_final['item_prod'].str.strip()

df_final['item_amount'] = pd.to_numeric(df_final['item_amount'], errors='coerce')

print(df_final[['item_amount', 'item_prod']].head())

   item_amount      item_prod
0          1.0    Potato cake
1          1.0            Tea
2          1.0  americano 0.2
2          1.0  americano 0.3
2          1.0  Carrot pieces


In [ ]:
# Save the cleaned data to a CSV file for import into DBeaver
cols_to_keep = [
    'Дата', 'Номер чека', 'Продажи', 'Скидки', 'Выручка', 'Налоги', 
    'Итого', 'Себестоимость товаров', 'Валовая прибыль', 
    'item_amount', 'item_prod', 'Касса'
]
df_final_clean = df_final[cols_to_keep]
df_final_clean.to_csv('data_for_dbeaver_v2.csv', index=False, sep=';', encoding='utf-8')
